# V6.3 Canonical Rebuild Pipeline

This notebook does **not** patch V6.2E in place.

It creates a controlled V6.3 rebuild workflow around the accepted V6.2E freeze:

1. reconstruct the exact V6.2E lineage;
2. preserve and recover as much provenance as the manifest/path layout exposes;
3. define one canonical bbox policy per class;
4. generate source × class review packs;
5. auto-detect exact/near-identical duplicate boxes;
6. accept an explicit `source_class_policy.csv` and optional `bbox_corrections.csv`;
7. refuse to freeze V6.3 while unresolved risky source/class groups remain;
8. build a deterministic candidate dataset only after the policy gate passes.

No model training is performed here. V6.2E remains untouched as the baseline.


In [ ]:
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, hashlib, zipfile, math, re
from collections import Counter, defaultdict

W = Path("/kaggle/working")
TMP = Path("/tmp/v6p3_rebuild_clean_v3")
BASE_EXTRACT = TMP/"base"
OVERLAY_EXTRACT = TMP/"overlay"
RECON_LABELS = TMP/"reconciled_labels"
OUT = W/"v6p3_rebuild"
CANDIDATE = W/"v6p3_candidate"

for p in [TMP, BASE_EXTRACT, OVERLAY_EXTRACT, RECON_LABELS, OUT]:
    p.mkdir(parents=True, exist_ok=True)

FINAL_TAG = "v6p2e_23562a45b343"
PARENT_TAG = "v6p1_5d0b1149891c"
BASE_TAG = "56b7ea194ee5"
OVERLAY_TAG = "ee2d3c4497fd"

DATA_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
FINAL_ROOT = f"{DATA_ROOT}/v6p2e/{FINAL_TAG}"
PARENT_ROOT = f"{DATA_ROOT}/v6p1/{PARENT_TAG}"

BASE_STASH = f"{DATA_ROOT}/curated_v6_rs1280_{BASE_TAG}.tar"
OVERLAY_STASH = f"{PARENT_ROOT}/v6p1_fotl_overlay_{OVERLAY_TAG}.tar.gz"
FREEZE_STASH = f"{FINAL_ROOT}/v6p2e.freeze.json"
MANIFEST_STASH = f"{FINAL_ROOT}/v6p2e_split_manifest.csv"
RECON_STASH = f"{FINAL_ROOT}/v6p2e_reconciled_labels.zip"

NAMES = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

RISK_CLASSES = {
    "insulator_broken",
    "insulator_flashover",
    "broken_strand",
    "bird_nest",
}

AUTO_DUP_IOU = 0.98
REVIEW_SAMPLE_PER_GROUP = 80
SEED = 42

CANONICAL_POLICY = {
    "insulator": {
        "target": "complete visible insulator / insulator string context",
        "bbox_rule": "tight box around the complete visible target; do not label a local defect region as insulator",
    },
    "insulator_broken": {
        "target": "localized structural defect",
        "bbox_rule": "box only the visible broken/chipped/missing-disc structural defect region, not the whole insulator string",
    },
    "insulator_flashover": {
        "target": "localized visible flashover damage",
        "bbox_rule": "box the visible damage region; do not box the whole insulator unless the visible damage genuinely spans it",
    },
    "bird_nest": {
        "target": "complete visible bird nest",
        "bbox_rule": "tight box around the nest itself, excluding unrelated tower/insulator context",
    },
    "foreign_object": {
        "target": "complete visible foreign object",
        "bbox_rule": "tight box around the foreign object itself",
    },
    "broken_strand": {
        "target": "localized visibly damaged/broken conductor strand segment",
        "bbox_rule": "box the damaged segment, not an arbitrarily long intact conductor section",
    },
}


## 1. Runtime + Camber

In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "pyyaml", "pillow"
])

import pandas as pd
import numpy as np
import yaml
from PIL import Image, ImageDraw, ImageOps

from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret("CAMBER_API_KEY")
if not key:
    raise RuntimeError("Missing CAMBER_API_KEY")
os.environ["CAMBER_API_KEY"] = key

if shutil.which("camber") is None and not (Path.home()/".camber/bin/camber").exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True
    )

os.environ["PATH"] = (
    f"{Path.home()}/.camber/bin:"
    f"{Path.home()}/.local/bin:"
    + os.environ.get("PATH","")
)
CAMBER = shutil.which("camber") or str(Path.home()/".camber/bin/camber")

def stash_cp(src, dst, required=True):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(
        [CAMBER, "stash", "cp", str(src), str(dst)],
        text=True, capture_output=True, env=os.environ.copy()
    )
    ok = r.returncode == 0 and dst.exists()
    if required and not ok:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    return ok

def stash_put(src, dst):
    r = subprocess.run(
        [CAMBER, "stash", "cp", str(src), str(dst)],
        text=True, capture_output=True, env=os.environ.copy()
    )
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    print("Uploaded:", dst)


## 2. Restore exact V6.2E

In [ ]:
freeze_local = TMP/"v6p2e.freeze.json"
manifest_local = TMP/"v6p2e_split_manifest.csv"
recon_zip = TMP/"v6p2e_reconciled_labels.zip"
base_tar = TMP/f"base_{BASE_TAG}.tar"
overlay_tar = TMP/f"overlay_{OVERLAY_TAG}.tar.gz"

# IMPORTANT:
# Never reuse previously extracted /tmp source trees. Earlier V6.3 notebook
# versions used "if data.yaml exists -> reuse", which can preserve a stale or
# partial extraction across reruns in the same Kaggle session.
for p in [BASE_EXTRACT, OVERLAY_EXTRACT, RECON_LABELS]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

for p in [freeze_local, manifest_local, recon_zip, base_tar, overlay_tar]:
    p.unlink(missing_ok=True)

print("Downloading accepted V6.2E metadata...")
stash_cp(FREEZE_STASH, freeze_local)
stash_cp(MANIFEST_STASH, manifest_local)
stash_cp(RECON_STASH, recon_zip)

freeze = json.loads(freeze_local.read_text())
manifest = pd.read_csv(manifest_local)

assert freeze["final_tag"] == FINAL_TAG
assert freeze["freeze_ready"] is True
assert len(manifest) == sum(freeze["split_counts"].values()), (
    len(manifest), freeze["split_counts"]
)

print("Downloading immutable source archives from Camber...")
stash_cp(BASE_STASH, base_tar)
stash_cp(OVERLAY_STASH, overlay_tar)

# Archive sanity before extraction.
def tar_names(path, gz=False):
    flag = "-tzf" if gz else "-tf"
    r = subprocess.run(
        ["tar", flag, str(path)],
        text=True, capture_output=True
    )
    if r.returncode != 0:
        raise RuntimeError(
            f"Cannot list archive {path}\n{r.stderr[-2000:]}"
        )
    return r.stdout.splitlines()

base_names = tar_names(base_tar, gz=False)
overlay_names = tar_names(overlay_tar, gz=True)

print("BASE archive entries:", len(base_names))
print("OVERLAY archive entries:", len(overlay_names))

# This exact file is the one that exposed stale/mismatched extraction earlier.
# Do not require the basename to exist because a previous curation stage may
# legitimately rename files; this is only a diagnostic.
_known_problem_basename = "fotl_image_val159_6530eaae.jpg"
print(
    "Known-problem basename present in raw OVERLAY archive:",
    any(Path(x).name == _known_problem_basename for x in overlay_names)
)

print("Extracting clean source trees...")
subprocess.check_call(["tar","-xf",str(base_tar),"-C",str(BASE_EXTRACT)])
subprocess.check_call(["tar","-xzf",str(overlay_tar),"-C",str(OVERLAY_EXTRACT)])

# Reconciled labels are also rebuilt cleanly.
with zipfile.ZipFile(recon_zip) as z:
    z.extractall(RECON_LABELS)

def norm_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x:int(x))]
    return list(names)

def find_root(root):
    matches=[]
    for yp in Path(root).rglob("data.yaml"):
        try:
            cfg = yaml.safe_load(yp.read_text())
            if norm_names(cfg.get("names",[])) == NAMES:
                matches.append(yp.parent)
        except Exception:
            pass
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one six-class dataset root under {root}; "
            f"found {len(matches)}: {matches}"
        )
    return matches[0]

BASE = find_root(BASE_EXTRACT)
OVERLAY = find_root(OVERLAY_EXTRACT)

print("Freeze:", FINAL_TAG)
print("Manifest rows:", len(manifest))
print("BASE:", BASE)
print("OVERLAY:", OVERLAY)
print("Manifest columns:", list(manifest.columns))

# Do not retain archive files after a successful extraction.
base_tar.unlink(missing_ok=True)
overlay_tar.unlink(missing_ok=True)
recon_zip.unlink(missing_ok=True)


## 3. Recover provenance

The previous audit collapsed most samples to `base` or `fotl`.
This cell tries to preserve more lineage from:

- explicit manifest columns, if present;
- the original source path;
- subdirectories under `base/` or `fotl/`.

If the old build flattened source identity completely, the notebook reports that limitation instead of inventing provenance.


### Clean-restore V3

Previous versions reused `/tmp/v6p3_rebuild` whenever a `data.yaml` was already
present. In a long-lived Kaggle session that can preserve a stale or partial extraction.

V3 never trusts that cache. It downloads the accepted V6.2E metadata and both immutable
source archives again, deletes the old extraction trees, extracts them cleanly, and only
then performs path/SHA recovery.

Therefore:

- if the previously missing FOTL image is recovered, the problem was stale extraction;
- if V3 still reports `CLEAN-RESTORE INTEGRITY FAILURE`, the Camber source artifacts
  themselves no longer contain bytes required by the accepted manifest, and V6.3 rebuild
  must stop until the missing source bytes are restored.


In [ ]:
PROVENANCE_COL_CANDIDATES = [
    "source_dataset", "original_source", "dataset", "dataset_source",
    "raw_source", "provenance", "source_name",
]

IMG_EXTS={".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

def sha256_file(path, chunk=4*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def explicit_provenance(row):
    for c in PROVENANCE_COL_CANDIDATES:
        if c in manifest.columns:
            v = str(row.get(c,"")).strip()
            if v and v.lower() not in {"nan","none",""}:
                return v, f"manifest:{c}"
    return None, None

def parse_old_path(row):
    old_path = str(row["image_path_source"]).replace("\\","/")
    old_split = str(row["previous_v6p1_split"])

    # Prefer the declared split marker, but tolerate rows whose logical split lineage
    # no longer matches the physical archive split.
    marker = f"/images/{old_split}/"
    if marker in old_path:
        suffix = old_path.split(marker,1)[1]
    else:
        m = re.search(r"/images/(train|val|test)/(.*)$", old_path)
        if not m:
            raise RuntimeError(f"Cannot parse source path: {old_path}")
        suffix = m.group(2)

    if suffix.startswith("base/"):
        family = "base"
        rel = suffix[len("base/"):]
        preferred_root = BASE
    elif suffix.startswith("fotl/"):
        family = "fotl"
        rel = suffix[len("fotl/"):]
        preferred_root = OVERLAY
    else:
        raise RuntimeError(f"Unknown source suffix: {suffix}")

    parts = Path(rel).parts
    path_group = parts[0] if len(parts) > 1 else family
    return old_path, old_split, family, rel, preferred_root, path_group

# Build cheap basename indexes once for BOTH immutable source artifacts.
ROOT_NAME = {
    str(BASE): "base",
    str(OVERLAY): "fotl",
}
ALL_ROOTS=[BASE,OVERLAY]

BASENAME_INDEX=defaultdict(list)
for rr in ALL_ROOTS:
    img_root=Path(rr)/"images"
    if not img_root.exists():
        continue
    for p in img_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            BASENAME_INDEX[p.name].append((Path(rr),p))

print("Indexed physical source images by basename:",
      sum(len(v) for v in BASENAME_INDEX.values()))

_SHA_INDEX=None

def build_sha_index():
    """
    Expensive fallback, built at most once.
    It scans both BASE and OVERLAY and maps SHA256 -> actual image path/root.
    This handles artifacts where a manifest's original filename/path is no longer
    present but the exact image bytes survived under a renamed path.
    """
    global _SHA_INDEX
    if _SHA_INDEX is not None:
        return _SHA_INDEX

    idx=defaultdict(list)
    n=0
    for rr in ALL_ROOTS:
        img_root=Path(rr)/"images"
        if not img_root.exists():
            continue
        for p in img_root.rglob("*"):
            if not (p.is_file() and p.suffix.lower() in IMG_EXTS):
                continue
            idx[sha256_file(p)].append((Path(rr),p))
            n+=1
            if n%5000==0:
                print("SHA-indexed",n,"source images...")

    _SHA_INDEX=idx
    print("✅ SHA index built:",n,"images,",len(idx),"unique hashes")
    return _SHA_INDEX

def label_for_actual_image(actual_root, ip, rel_hint=None):
    actual_root=Path(actual_root)
    ip=Path(ip)

    try:
        rel_from_images=ip.relative_to(actual_root/"images")
        lp=(actual_root/"labels"/rel_from_images).with_suffix(".txt")
        if lp.exists():
            return lp, str(rel_from_images.parts[0])
    except Exception:
        rel_from_images=None

    # Fallback by basename.
    candidates=list((actual_root/"labels").rglob(ip.stem+".txt"))
    if rel_hint is not None:
        hint_name=Path(rel_hint).with_suffix(".txt").name
        hinted=[p for p in candidates if p.name==hint_name]
        if len(hinted)==1:
            candidates=hinted

    if len(candidates)==1:
        lp=candidates[0]
        try:
            physical_split=str(lp.relative_to(actual_root/"labels").parts[0])
        except Exception:
            physical_split="unknown"
        return lp, physical_split

    # We still return a derived path if possible so caller can emit a precise error.
    if rel_from_images is not None:
        return (actual_root/"labels"/rel_from_images).with_suffix(".txt"), str(rel_from_images.parts[0])

    return actual_root/"labels"/(ip.stem+".txt"), "unknown"

def resolve_physical_pair(preferred_root, old_split, rel, expected_sha):
    """
    Resolve exact source bytes with progressively broader search:

      A. preferred root + declared split
      B. preferred root + any split
      C. same basename across BOTH BASE and OVERLAY
      D. exact SHA256 across BOTH BASE and OVERLAY, even if renamed

    A file is accepted only when SHA256 == manifest SHA256.
    Returns:
      actual_root, image_path, label_path, physical_source_split, resolution_mode
    """
    preferred_root=Path(preferred_root)
    rel=Path(rel)

    candidates=[]
    seen=set()

    def add(actual_root,p,mode):
        actual_root=Path(actual_root)
        p=Path(p)
        key=(str(actual_root),str(p))
        if key not in seen:
            seen.add(key)
            candidates.append((actual_root,p,mode))

    # A: exact manifest lineage path.
    add(preferred_root, preferred_root/"images"/old_split/rel, "preferred_declared_split")

    # B: same relative path in any split of preferred root.
    for sp in ["train","val","test"]:
        add(preferred_root, preferred_root/"images"/sp/rel, "preferred_other_split")

    # C: basename across both artifacts.
    for actual_root,p in BASENAME_INDEX.get(rel.name,[]):
        add(actual_root,p,"cross_root_same_basename")

    existing=[(rr,p,mode) for rr,p,mode in candidates if p.exists() and p.is_file()]

    for actual_root,ip,mode in existing:
        if sha256_file(ip)==expected_sha:
            lp,physical_split=label_for_actual_image(actual_root,ip,rel)
            return actual_root,ip,lp,physical_split,mode

    # D: Exact-byte fallback across both artifacts.
    sha_index=build_sha_index()
    matches=sha_index.get(expected_sha,[])
    if matches:
        # Prefer the manifest-indicated artifact if the hash exists in both.
        matches=sorted(matches,key=lambda x: 0 if Path(x[0])==preferred_root else 1)
        actual_root,ip=matches[0]
        lp,physical_split=label_for_actual_image(actual_root,ip,rel)
        return actual_root,ip,lp,physical_split,"cross_root_sha256"

    # At this point we have freshly downloaded + freshly extracted both immutable
    # source archives in this same run, and searched them by exact SHA256.
    # Therefore this is a genuine artifact/manifest mismatch, not a path-cache issue.
    missing_row = {
        "expected_sha": expected_sha,
        "preferred_root": str(preferred_root),
        "old_split": old_split,
        "relative_path": str(rel),
        "same_basename_candidates": [
            str(p) for _,p in BASENAME_INDEX.get(rel.name,[])[:20]
        ],
    }
    diag = OUT/"MISSING_SOURCE_BYTES.json"
    diag.write_text(json.dumps(missing_row, indent=2))
    raise FileNotFoundError(
        "CLEAN-RESTORE INTEGRITY FAILURE.\n"
        "The exact manifest SHA256 is absent from BOTH freshly restored source artifacts.\n"
        f"Diagnostic written to: {diag}\n"
        + json.dumps(missing_row, indent=2)
    )

rows=[]

for idx,row in manifest.iterrows():
    old_path,old_split,family,rel,preferred_root,path_group=parse_old_path(row)
    expected_sha=str(row["sha256"])

    prov,prov_basis=explicit_provenance(row)
    if prov is None:
        if path_group not in {"base","fotl"} and len(Path(rel).parts)>1:
            prov=path_group
            prov_basis="path_subdir"
        else:
            prov=family
            prov_basis="coarse_family_only"

    actual_root,ip,original_lp,physical_source_split,resolution_mode=resolve_physical_pair(
        preferred_root=preferred_root,
        old_split=old_split,
        rel=rel,
        expected_sha=expected_sha,
    )

    # If the manifest says this sample used a reconciled label, that label wins.
    rp=RECON_LABELS/f"{expected_sha}.txt"
    if str(row.get("resolution","")).startswith("merged"):
        if not rp.exists():
            raise FileNotFoundError(f"Missing reconciled label: {rp}")
        lp=rp
    else:
        lp=original_lp
        if not lp.exists():
            raise FileNotFoundError(
                "Resolved exact image bytes but corresponding label is missing.\n"
                f"actual_root={actual_root}\n"
                f"image={ip}\n"
                f"derived_label={lp}\n"
                f"sha={expected_sha}\n"
                f"resolution_mode={resolution_mode}"
            )

    rows.append({
        "row_id":idx,
        "sha256":expected_sha,
        "split":str(row["split"]),
        "previous_split":old_split,
        "physical_source_split":physical_source_split,
        "coarse_family":family,
        "actual_artifact":ROOT_NAME.get(str(actual_root),str(actual_root)),
        "provenance":str(prov),
        "provenance_basis":prov_basis,
        "relative_path":rel,
        "image_path":str(ip),
        "label_path":str(lp),
        "path_resolution_mode":resolution_mode,
        "resolution":str(row.get("resolution","")),
    })

rec=pd.DataFrame(rows)

if (rec.provenance_basis=="coarse_family_only").mean()>0.80:
    print("⚠️ Most original dataset provenance is no longer recoverable from this artifact.")
    print("   V6.3 can still canonicalize boxes, but source-by-source conclusions will be limited.")

print("\nResolution modes:")
display(rec["path_resolution_mode"].value_counts().rename_axis("mode").reset_index(name="images"))

cross_artifact=rec[
    ((rec.coarse_family=="fotl") & (rec.actual_artifact!="fotl")) |
    ((rec.coarse_family=="base") & (rec.actual_artifact!="base"))
]
print("Cross-artifact recoveries:",len(cross_artifact))
if len(cross_artifact):
    display(cross_artifact[
        ["coarse_family","actual_artifact","previous_split",
         "physical_source_split","relative_path","path_resolution_mode"]
    ].head(50))

split_mismatch=rec[rec["previous_split"]!=rec["physical_source_split"]]
print("Physical split relocations resolved:",len(split_mismatch))
if len(split_mismatch):
    display(split_mismatch[
        ["coarse_family","actual_artifact","previous_split",
         "physical_source_split","relative_path","path_resolution_mode"]
    ].head(50))

display(
    rec.groupby(["provenance","provenance_basis","split"])
       .size().reset_index(name="images")
       .sort_values("images",ascending=False)
       .head(100)
)

rec.to_csv(OUT/"recovered_provenance.csv",index=False)


## 4. Parse labels

In [ ]:
def parse_label(path):
    boxes = []
    for line_no,line in enumerate(Path(path).read_text(errors="ignore").splitlines(),1):
        if not line.strip():
            continue
        p = line.split()
        if len(p) < 5:
            raise RuntimeError(f"Malformed label: {path}:{line_no}")
        cid = int(float(p[0]))
        x,y,w,h = map(float,p[1:5])
        boxes.append({
            "line_no":line_no,
            "class_id":cid,
            "class":NAMES[cid],
            "cx":x,"cy":y,"w":w,"h":h,
        })
    return boxes

box_rows = []
image_rows = []

for _,r in rec.iterrows():
    boxes = parse_label(r.label_path)
    image_rows.append({**r.to_dict(), "n_boxes":len(boxes)})

    with Image.open(r.image_path) as im:
        Wd,Hd = im.size

    for b in boxes:
        area = b["w"]*b["h"]
        ar = max(b["w"]/b["h"], b["h"]/b["w"]) if b["w"]>0 and b["h"]>0 else np.inf
        box_rows.append({
            **r.to_dict(),
            **b,
            "area":area,
            "aspect_ratio":ar,
            "pixel_w":b["w"]*Wd,
            "pixel_h":b["h"]*Hd,
        })

images_df = pd.DataFrame(image_rows)
boxes_df = pd.DataFrame(box_rows)

print("Images:",len(images_df))
print("Boxes:",len(boxes_df))
display(boxes_df.groupby(["provenance","class"]).size().reset_index(name="boxes"))


## 5. Canonical bbox policy

These rules are the acceptance criteria for V6.3.
They are written to disk so the dataset version is tied to a documented annotation policy.


In [ ]:
policy_path = OUT/"v6p3_bbox_policy.json"
policy_path.write_text(json.dumps(CANONICAL_POLICY,ensure_ascii=False,indent=2))

for cls,rule in CANONICAL_POLICY.items():
    print("\n",cls)
    print(" target   :",rule["target"])
    print(" bbox rule:",rule["bbox_rule"])


## 6. Source × class geometry and review priority

The goal is not to automatically call a source wrong.
The goal is to find source/class groups whose box geometry differs enough to deserve visual review.


In [ ]:
global_med = boxes_df.groupby("class").area.median().to_dict()

group_rows = []
for (prov,cls),g in boxes_df.groupby(["provenance","class"]):
    med = float(g.area.median())
    gm = float(global_med[cls])
    ratio = med/gm if gm>0 else np.nan

    group_rows.append({
        "provenance":prov,
        "class":cls,
        "boxes":len(g),
        "images":g.sha256.nunique(),
        "area_median":med,
        "area_q10":float(g.area.quantile(.10)),
        "area_q90":float(g.area.quantile(.90)),
        "median_area_ratio_to_class_global":ratio,
        "pixel_w_median":float(g.pixel_w.median()),
        "pixel_h_median":float(g.pixel_h.median()),
        "risk_class":cls in RISK_CLASSES,
        "geometry_outlier":bool(len(g)>=20 and (ratio<0.40 or ratio>2.50)),
    })

group_df = pd.DataFrame(group_rows).sort_values(["risk_class","geometry_outlier","class","boxes"],
                                                ascending=[False,False,True,False])

display(group_df)
group_df.to_csv(OUT/"source_class_geometry.csv",index=False)


## 7. Auto-detect near-identical duplicate boxes

Only very high-IoU duplicates of the **same class** are eligible for automatic removal.
The original V6.2E labels are never modified.


In [ ]:
def xywh_to_xyxy(x,y,w,h):
    return x-w/2,y-h/2,x+w/2,y+h/2

def iou(a,b):
    ax1,ay1,ax2,ay2 = xywh_to_xyxy(*a)
    bx1,by1,bx2,by2 = xywh_to_xyxy(*b)
    ix1=max(ax1,bx1); iy1=max(ay1,by1)
    ix2=min(ax2,bx2); iy2=min(ay2,by2)
    iw=max(0.0,ix2-ix1); ih=max(0.0,iy2-iy1)
    inter=iw*ih
    aa=max(0.0,ax2-ax1)*max(0.0,ay2-ay1)
    bb=max(0.0,bx2-bx1)*max(0.0,by2-by1)
    den=aa+bb-inter
    return inter/den if den>0 else 0.0

dup_rows=[]

for sha,g in boxes_df.groupby("sha256"):
    rr=g.sort_values("line_no").to_dict("records")
    for i in range(len(rr)):
        for j in range(i+1,len(rr)):
            a,b=rr[i],rr[j]
            if a["class_id"] != b["class_id"]:
                continue
            ov=iou(
                (a["cx"],a["cy"],a["w"],a["h"]),
                (b["cx"],b["cy"],b["w"],b["h"])
            )
            if ov>=AUTO_DUP_IOU:
                dup_rows.append({
                    "sha256":sha,
                    "class":a["class"],
                    "keep_line_no":a["line_no"],
                    "drop_line_no":b["line_no"],
                    "iou":ov,
                    "image_path":a["image_path"],
                    "label_path":a["label_path"],
                })

dup_df=pd.DataFrame(dup_rows)
display(dup_df)
dup_df.to_csv(OUT/"auto_duplicate_box_candidates.csv",index=False)
print("Auto duplicate candidates:",len(dup_df))


## 8. Generate visual source/class review pack

Review a sample per risky source/class group before making any global source policy decision.


In [ ]:
rng=np.random.default_rng(SEED)
REVIEW_DIR=OUT/"review_sheets"
REVIEW_DIR.mkdir(parents=True,exist_ok=True)

def draw_boxes(path, rows, thumb=(360,260)):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert("RGB")
        Wd,Hd=im.size
        d=ImageDraw.Draw(im)
        for r in rows:
            x,y,w,h=r["cx"],r["cy"],r["w"],r["h"]
            x1=(x-w/2)*Wd; y1=(y-h/2)*Hd
            x2=(x+w/2)*Wd; y2=(y+h/2)*Hd
            d.rectangle([x1,y1,x2,y2],width=max(2,int(min(Wd,Hd)/350)))
            d.text((x1,max(0,y1-13)),r["class"])
        im.thumbnail(thumb)
        canvas=Image.new("RGB",thumb)
        canvas.paste(im,((thumb[0]-im.width)//2,(thumb[1]-im.height)//2))
        return canvas

review_rows=[]

for _,gr in group_df.iterrows():
    prov,cls=gr["provenance"],gr["class"]
    if cls not in RISK_CLASSES:
        continue

    gg=boxes_df[(boxes_df.provenance==prov)&(boxes_df["class"]==cls)]
    shas=gg.sha256.drop_duplicates().to_numpy()

    if len(shas)==0:
        continue

    take=min(REVIEW_SAMPLE_PER_GROUP,len(shas))
    chosen=rng.choice(shas,size=take,replace=False)

    for sha in chosen:
        row=gg[gg.sha256==sha].iloc[0]
        review_rows.append({
            "provenance":prov,
            "class":cls,
            "sha256":sha,
            "image_path":row.image_path,
            "policy_decision":"",
            "review_note":"",
        })

review_df=pd.DataFrame(review_rows)
review_df.to_csv(OUT/"visual_review_manifest.csv",index=False)

# Make contact sheets by source/class.
for (prov,cls),g in review_df.groupby(["provenance","class"]):
    ims=[]
    for sha in g.sha256.tolist():
        all_boxes=boxes_df[boxes_df.sha256==sha]
        if all_boxes.empty: continue
        ims.append(draw_boxes(
            all_boxes.iloc[0].image_path,
            all_boxes.to_dict("records")
        ))

    if not ims: continue
    cols=5
    rows=math.ceil(len(ims)/cols)
    tw,th=360,260
    sheet=Image.new("RGB",(cols*tw,rows*th))
    for i,im in enumerate(ims):
        sheet.paste(im,((i%cols)*tw,(i//cols)*th))

    safe=re.sub(r"[^A-Za-z0-9_.-]+","_",f"{prov}__{cls}")
    sheet.save(REVIEW_DIR/f"{safe}.jpg",quality=90)

print("Review samples:",len(review_df))
print("Sheets:",len(list(REVIEW_DIR.glob("*.jpg"))))


## 9. Create source/class policy template

Allowed decisions:

- `KEEP`
- `EXCLUDE`
- `RELABEL_REQUIRED`

Do not mark a group `KEEP` unless its sampled boxes follow the canonical rule.
`RELABEL_REQUIRED` blocks V6.3 freezing until corrected labels are supplied.


In [ ]:
policy_rows=[]

for _,r in group_df.iterrows():
    default=""
    if r["class"] not in RISK_CLASSES and not r["geometry_outlier"]:
        default="KEEP"

    policy_rows.append({
        "provenance":r["provenance"],
        "class":r["class"],
        "boxes":r["boxes"],
        "images":r["images"],
        "geometry_outlier":r["geometry_outlier"],
        "decision":default,
        "review_note":"",
    })

policy_template=pd.DataFrame(policy_rows)
policy_template.to_csv(OUT/"source_class_policy_TEMPLATE.csv",index=False)
display(policy_template)


## 10. Optional bbox correction template

Use this only for individual labels that must be fixed.

Actions:

- `KEEP`
- `DROP`
- `REPLACE`

For `REPLACE`, fill `new_class_id,new_cx,new_cy,new_w,new_h`.


In [ ]:
corr_cols=[
    "sha256","line_no","action",
    "new_class_id","new_cx","new_cy","new_w","new_h",
    "reason"
]
pd.DataFrame(columns=corr_cols).to_csv(
    OUT/"bbox_corrections_TEMPLATE.csv",index=False
)
print("Created:",OUT/"bbox_corrections_TEMPLATE.csv")


## 11. Resolve policy/corrections input

Attach reviewed files to Kaggle with names:

- `source_class_policy.csv`
- optional `bbox_corrections.csv`

The notebook searches `/kaggle/input` recursively.
If no reviewed policy exists, it stops after uploading the review pack.


In [ ]:
def find_input_file(name):
    base=Path("/kaggle/input")
    if not base.exists():
        return None
    hits=list(base.rglob(name))
    return hits[0] if hits else None

reviewed_policy_path=find_input_file("source_class_policy.csv")
corrections_path=find_input_file("bbox_corrections.csv")

print("Reviewed policy:",reviewed_policy_path)
print("Corrections:",corrections_path)


## 12. Persist review pack before any rebuild

In [ ]:
REMOTE_REVIEW=(
    f"stash://bunpmc/projects/yolov11sdi/reports/v6p3_rebuild/"
    f"from_{FINAL_TAG}"
)

review_zip=W/f"{FINAL_TAG}_v6p3_review_pack.zip"

with zipfile.ZipFile(review_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(OUT.parent))

stash_put(review_zip,f"{REMOTE_REVIEW}/{review_zip.name}")
print("Review pack uploaded.")


## 13. Hard gate before V6.3 build

This is intentionally strict.

The notebook refuses to build V6.3 when:

- no reviewed source/class policy exists;
- a policy decision is blank;
- any `RELABEL_REQUIRED` group remains unresolved.

This prevents another "we merged it and hoped the bbox semantics matched" dataset version.


In [ ]:
if reviewed_policy_path is None:
    raise RuntimeError(
        "STOP AFTER REVIEW PACK. Review source_class_policy_TEMPLATE.csv and the contact sheets, "
        "then attach the completed file as source_class_policy.csv and rerun from this cell."
    )

policy=pd.read_csv(reviewed_policy_path).fillna("")
required={"provenance","class","decision"}
if not required <= set(policy.columns):
    raise RuntimeError(f"Policy missing columns: {required-set(policy.columns)}")

allowed={"KEEP","EXCLUDE","RELABEL_REQUIRED"}
bad=policy[~policy.decision.isin(allowed)]
if len(bad):
    display(bad)
    raise RuntimeError("Unresolved/invalid source-class policy decisions remain.")

pending=policy[policy.decision=="RELABEL_REQUIRED"]
if len(pending) and corrections_path is None:
    display(pending)
    raise RuntimeError(
        "RELABEL_REQUIRED groups exist but bbox_corrections.csv was not supplied."
    )

display(policy)
print("✅ Policy gate passed.")


## 14. Build deterministic V6.3 candidate

Rules:

1. V6.2E original files remain untouched.
2. `EXCLUDE` source/class boxes are removed.
3. same-class duplicate boxes with IoU ≥ 0.98 are deduplicated;
4. explicit bbox corrections override the original annotation;
5. every change is logged;
6. split membership remains unchanged in this candidate.


In [ ]:
if CANDIDATE.exists():
    shutil.rmtree(CANDIDATE)

for sp in ["train","val","test"]:
    (CANDIDATE/"images"/sp).mkdir(parents=True,exist_ok=True)
    (CANDIDATE/"labels"/sp).mkdir(parents=True,exist_ok=True)

decision={(str(r.provenance),str(r["class"])):str(r.decision)
          for _,r in policy.iterrows()}

corrections={}
if corrections_path is not None:
    cdf=pd.read_csv(corrections_path).fillna("")
    for _,r in cdf.iterrows():
        corrections[(str(r.sha256),int(r.line_no))]=r.to_dict()

dup_drop=set()
for _,r in dup_df.iterrows():
    dup_drop.add((str(r.sha256),int(r.drop_line_no)))

change_log=[]

for i,r in images_df.iterrows():
    sp=r["split"]
    src=Path(r["image_path"])
    ext=src.suffix.lower()
    name=f"{r['sha256'][:16]}_{i:06d}{ext}"
    dst_img=CANDIDATE/"images"/sp/name
    dst_lab=CANDIDATE/"labels"/sp/Path(name).with_suffix(".txt")

    try:
        os.link(src,dst_img)
    except Exception:
        shutil.copy2(src,dst_img)

    orig=parse_label(r["label_path"])
    final=[]

    for b in orig:
        key_group=(r["provenance"],b["class"])
        dec=decision.get(key_group,"")
        key_box=(r["sha256"],b["line_no"])

        if dec=="EXCLUDE":
            change_log.append({
                "sha256":r["sha256"],"line_no":b["line_no"],
                "action":"DROP_SOURCE_CLASS_POLICY","class":b["class"],
                "provenance":r["provenance"],
            })
            continue

        if key_box in dup_drop:
            change_log.append({
                "sha256":r["sha256"],"line_no":b["line_no"],
                "action":"DROP_AUTO_DUPLICATE","class":b["class"],
                "provenance":r["provenance"],
            })
            continue

        corr=corrections.get(key_box)
        if corr:
            action=str(corr.get("action","")).upper()
            if action=="DROP":
                change_log.append({
                    "sha256":r["sha256"],"line_no":b["line_no"],
                    "action":"DROP_MANUAL","class":b["class"],
                    "provenance":r["provenance"],
                })
                continue
            elif action=="REPLACE":
                nb={
                    "class_id":int(float(corr["new_class_id"])),
                    "cx":float(corr["new_cx"]),
                    "cy":float(corr["new_cy"]),
                    "w":float(corr["new_w"]),
                    "h":float(corr["new_h"]),
                }
                nb["class"]=NAMES[nb["class_id"]]
                final.append(nb)
                change_log.append({
                    "sha256":r["sha256"],"line_no":b["line_no"],
                    "action":"REPLACE_MANUAL",
                    "class_before":b["class"],"class_after":nb["class"],
                    "provenance":r["provenance"],
                })
                continue

        final.append(b)

    with open(dst_lab,"w") as f:
        for b in final:
            f.write(
                f"{int(b['class_id'])} "
                f"{float(b['cx']):.8f} {float(b['cy']):.8f} "
                f"{float(b['w']):.8f} {float(b['h']):.8f}\n"
            )

pd.DataFrame(change_log).to_csv(OUT/"v6p3_change_log.csv",index=False)

(CANDIDATE/"data.yaml").write_text(yaml.safe_dump({
    "path":str(CANDIDATE),
    "train":"images/train",
    "val":"images/val",
    "test":"images/test",
    "names":NAMES,
},sort_keys=False))

print("Candidate built:",CANDIDATE)
print("Changes:",len(change_log))


## 15. Candidate integrity + freeze metadata

In [ ]:
def parse_yolo(path):
    out=[]
    for ln,line in enumerate(Path(path).read_text(errors="ignore").splitlines(),1):
        if not line.strip(): continue
        p=line.split()
        if len(p)<5:
            raise RuntimeError(f"Malformed {path}:{ln}")
        cid=int(float(p[0]))
        x,y,w,h=map(float,p[1:5])
        if not (0<=cid<len(NAMES)):
            raise RuntimeError(f"Bad class {cid}: {path}:{ln}")
        if not (0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1):
            raise RuntimeError(f"Bad bbox {x,y,w,h}: {path}:{ln}")
        out.append((cid,x,y,w,h))
    return out

split_counts={}
box_counts={}
image_class_counts={}

for sp in ["train","val","test"]:
    imgs=list((CANDIDATE/"images"/sp).glob("*"))
    labs=list((CANDIDATE/"labels"/sp).glob("*.txt"))
    split_counts[sp]=len(imgs)

    bc=[0]*len(NAMES)
    ic=[0]*len(NAMES)

    for lp in labs:
        boxes=parse_yolo(lp)
        seen=set()
        for b in boxes:
            bc[b[0]]+=1
            seen.add(b[0])
        for cid in seen:
            ic[cid]+=1

    box_counts[sp]=bc
    image_class_counts[sp]=ic

print("Split counts:",split_counts)
print("Box counts:",box_counts)
print("Image class counts:",image_class_counts)

freeze_candidate={
    "parent_dataset_tag":FINAL_TAG,
    "status":"CANDIDATE_NOT_FINAL_UNTIL_YOLO_SANITY_PASS",
    "canonical_policy_version":"v6p3-policy-1",
    "split_counts":split_counts,
    "box_counts":box_counts,
    "image_class_counts":image_class_counts,
    "change_log_rows":len(change_log),
    "policy_rows":len(policy),
}

(OUT/"v6p3_candidate.freeze.json").write_text(
    json.dumps(freeze_candidate,indent=2)
)

candidate_zip=W/"v6p3_candidate_dataset.zip"
with zipfile.ZipFile(candidate_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for p in CANDIDATE.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(CANDIDATE.parent))

print(json.dumps(freeze_candidate,indent=2))
print("Candidate archive:",candidate_zip)


## 16. Upload candidate artifacts

The candidate is deliberately marked **NOT FINAL**.
Freeze V6.3 only after a fresh YOLO sanity run satisfies the agreed gate.


In [ ]:
REMOTE_CANDIDATE=(
    "stash://bunpmc/projects/yolov11sdi/datasets/"
    f"v6p3_candidate/from_{FINAL_TAG}"
)

stash_put(W/"v6p3_candidate_dataset.zip",
          f"{REMOTE_CANDIDATE}/v6p3_candidate_dataset.zip")
stash_put(OUT/"v6p3_candidate.freeze.json",
          f"{REMOTE_CANDIDATE}/v6p3_candidate.freeze.json")
stash_put(OUT/"v6p3_change_log.csv",
          f"{REMOTE_CANDIDATE}/v6p3_change_log.csv")
stash_put(OUT/"v6p3_bbox_policy.json",
          f"{REMOTE_CANDIDATE}/v6p3_bbox_policy.json")

print("✅ V6.3 candidate uploaded, NOT FINAL.")
